In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import torch.optim as optim
import gymnasium as gym
import numpy as np


In [3]:
class ReplayBuffer:
    def __init__(self, capacity, obs_dim):
        self.capacity = capacity
        self.step = 0

        self.states = np.zeros((capacity, obs_dim))
        self.actions = np.zeros(capacity, dtype=np.int8)
        self.rewards = np.zeros(capacity)
        self.next_states = np.zeros((capacity, obs_dim))
        self.terminations = np.zeros(capacity)
        self.truncations = np.zeros(capacity)
    
    def push(self, state, action, reward, next_state, termination, truncation):
        i = self.step
        self.states[i] = state
        self.actions[i] = action
        self.rewards[i] = reward
        self.next_states[i] = next_state
        self.terminations[i] = termination
        self.truncations[i] = truncation
        self.step = (self.step + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = np.random.randint(0, self.size, size=batch_size)
        return (self.states[idx], self.actions[idx], self.rewards[idx],
                self.next_states[idx], self.terminations[idx], self.truncations[idx])


In [4]:
class QNetwork(nn.Module):
    def __init__(self, s_dim, a_dim):
        super(QNetwork, self).__init__()

        self.layer1 = nn.Linear(s_dim, 100)
        self.layer2 = nn.Linear(100, 100)
        self.layer3 = nn.Linear(100, a_dim)

    def forward(self, x, target=None):
        if target==None:
            loss=None
        else:
            x = self.layer1(x)
            x = nn.ReLU(x)
            x = self.layer2(x)
            x = nn.ReLU(x)
            x = self.layer3(x)
            loss = F.smooth_l1_loss(x, target)
        return x, loss
      

In [ ]:
import random
from copy import deepcopy

class Agent:
    def __init__(self, env):
        self.initial_e = 1
        self.final_e = 0.05
        self.e_decay = 0.01
        self.lr = 0.00001
        self.reward_f = 0.97
        self.n_rollouts = 100
        self.n_iterations = 100
        self.batch_size = 32
        self.n_epochs = 100
        self.target_update_freq = 100
        self.env = env
        self.val_network = QNetwork(env.observation_space.shape[0], env.action_space.n)
        self.target_network = deepcopy(self.val_network)
        self.buffer = ReplayBuffer(1000000, env.observation_space.shape[0])
        self.total_steps = 0



    def greedy(self, state):
        state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        self.val_network.eval()
        with torch.no_grad():
            q_values: torch.tensor = self.val_network(state)
            return q_values.argmax().item()

    def epsilon_greedy(self, state):
        random_num = random.uniform(0,1)

        if random_num > self.initial_e:
            action = self.greedy(state)
        else:
            action = self.env.action_space.sample()

        return action


    def rollout(self):
        state, info = self.env.reset()
        k = 0
        while k < self.n_rollouts:
            action = self.epsilon_greedy(state)
            new_state, reward, terminated, truncated, info = self.env.step(action)
            self.buffer.push(state, action, reward, new_state, terminated, truncated)

            



In [5]:
env = gym.make("Acrobot-v1")
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<AcrobotEnv<Acrobot-v1>>>>>

6